In [1]:
import os
import sys

thisdir = os.path.dirname(os.path.realpath(""))
sys.path.append(os.path.join(thisdir, "CryptographicEstimators"))

In [2]:
import csv
proper_primes = None
with open('../assets/proper_primes.csv', 'r', newline='') as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        proper_primes = list(map(int, row))

In [4]:
from cryptographic_estimators.SDEstimator import SDEstimator,BJMM,BJMMpdw,BJMMplus,BJMMdw,BothMay

MEM_CONST = 0
MEM_LOG = 1
MEM_SQRT = 2
MEM_CBRT = 3

n_0=2
p=13232
t=134
#skip_algos= [BJMM,BJMMpdw,BJMMplus,BJMMdw,BothMay]
`2skip_algos= []
# assuming ISD(n,r,t)
def isd_compute(n,r,t):
    # SDEstimator requires n,k,t as params
    sd = SDEstimator(n,n-r,t, 
                 excluded_algorithms = skip_algos,
                 memory_access=MEM_LOG)
    res = sd.estimate()
    min_time = min(results.items(), key=lambda algo: algo['estimate']['time'])

    # best = { "name": "Prange",
    #          "time" : res["Prange"]["estimate"]["time"], 
    #          "memory": res["Prange"]["estimate"]["memory"]}
    # for algo in res:
    #     if best["time"] > res[algo]["estimate"]["time"]:
    #         best["name"] = algo
    #         best["time"] = res[algo]["estimate"]["time"]
    #         best["memory"] = res[algo]["estimate"]["memory"]
    #print(f'Best algorithm {best}')
    res["best"] = best
    print(res)
    return res["best"]["time"]

def isd_log(n,r,t):
    print(f"n:{n},r:{r},t:{t}")
    return n

# log required ISD or compute them
isd=isd_log

# message recovery ISD


# key recovery attack 1
def isd_kra_1(n_0,p,v):
    # ISD(n_0*p,p,2*v) / (p* binom{n_0}{2}), attacked code rate (1/n_0)
    logcost = isd(n_0*p,p)
    return logcost - log(p*binom(n_0,2),2).n()

def isd_kra_2(n_0,p,v):
    # ISD(2*p,p,2*v) / (n_0*p), attacked code rate (1/2)
    logcost = isd(2*p,p,2*v)
    return logcost - log(p*n_0,2).n()

def isd_kra_3(n_0,p,v):
    # ISD(n_0*p,(n_0-1)*p,n_0*v) / p
    logcost = isd(n_0*p,(n_0-1)*p,n_0*v)
    return logcost - log(p,2).n()

import json

def checkpoint_load(ckp_path):
    with open(ckp_path,"r") as infile:
        json_glob = infile.read()
        state_old = json.loads(json_glob)
    return state_old

def checkpoint_store(ckp_path,state):
    with open(ckp_path,"w") as outfile:
        outfile.write(json.dumps(state,sort_keys=True, indent=4))
    return    

# merges a json checkpoint. assumption: the checkpoint is
# a python dictionary indexed by (n,r,t)
def checkpoint_merge(ckp_path,state_to_merge):
    state_old = checkpoint_load(ckp_path)
    state = state_old | state_to_merge
    checkpoint_store(ckp_path,state)
    return  

# range creation
v_ranges
t_ranges = {128:[120,140], 192:[190,205]}


In [2]:
# Parallel processing stubs

def comp_load(x,y): 
    return x-y

import sage.parallel.multiprocessing_sage
n_parallel_proc = 4

#small helper to prepare an input dictionary for sage.parallel
def sage_parallel_input_prep(input_dict):
    return ( (), input_dict )

inputs = [{'x':2 , 'y':3},
          {'x':2 , 'y':3}]

work_batch = []
for el in inputs:
    work_batch.append(sage_parallel_input_prep(el))

cpus=2
parallel_iter = sage.parallel.multiprocessing_sage.parallel_iter(cpus,  comp_load, work_batch)
#par_results = list(parallel_iter)
for a in parallel_iter:
    print(a)
    print(a[0][1],a[1])

(((), {'x': 2, 'y': 3}), -1)
{'x': 2, 'y': 3} -1
(((), {'x': 2, 'y': 3}), -1)
{'x': 2, 'y': 3} -1


In [55]:
# Computation plan
 # build parameter space work batch
  # weight of the error/codeword should be in lambda pm20

 # each parallel worker analyzes a single parameter set
 # create a global ISD estimates dictionary:
 # create a set of all the tuples to be computed
 # construct parallel iterator on them, 
 # fill the dictionary in batches
 # call the memoized ISD function and derive parameters


[(((2,), {}), 4), (((3,), {}), 9)]

In [5]:
n,r,t = 120,60,6
foo=isd(n,r,t)
best = { "name": "Prange",
         "time" : foo["Prange"]["estimate"]["time"], 
         "memory": foo["Prange"]["estimate"]["memory"]}
for algo in foo:
    if best["time"] > foo[algo]["estimate"]["time"]:
        best["name"] = algo
        best["time"] = foo[algo]["estimate"]["time"]
        best["memory"] = foo[algo]["estimate"]["memory"]
print(f'Best algorithm {best}')
foo["best"] = best

memoized_isds={}

# jsonification does not allow dictionaries with tuples as indexes
dict_index = (n,r,t).__repr__()
memoized_isds[dict_index]=foo
print(memoized_isds)
(n,r,t).__repr__()

n:120,r:60,t:6


TypeError: 'sage.rings.integer.Integer' object is not subscriptable

In [118]:
# checkpointing tests
l = []
checkpoint_store(ckp_path,l)
for i in range(4):
    checkpoint_merge(ckp_path,l)
    l = l + [i]
print(checkpoint_load(ckp_path))
checkpoint_store("./test.json",memoized_isds)

[0, 0, 1, 0, 1, 2]


In [1]:
# Building parameter space
import proper_primes as pp
pp.proper_primes[0]

# approximate ISD hardness given the weight $w$ of the codeword/error to be found
# and the code rate k/n=R
# an ISD costs approx 2^cw, where the constant c depends on the rate c = log_2(1/(1-R))
# c = -log_2(1-R) = -log2(1-(n_0-1)/n_0)
# c_X is the constant for a given number of blocks n_0, and a parity check matrix H
# which is one block high and n_0 wide, hence R=(n_0-1)/n_0
# c_2 = 1
# c_3 = 1.58
# c_4 = 2



# sample rough trends for an ISD
for lam in [128,192,256]:
    for n_0 in range(2,5):
        # ISD(n_0*p,p,t) / sqrt(p) attacked code rate (n_0-1)/n_0 
        t = ceil( -lam / log(1-(n_0-1)/n_0,2) )
        # ISD(n_0*p,p,2*v) / (p* binom{n_0}{2}), attacked code rate (n_0-1)/n_0
        v_1 = ceil( -lam / log(1-(n_0-1)/n_0,2) ) // 2
        # ISD(2*p,p,2*v) / (n_0*p), attacked code rate (1/2)
        v_2 = ceil( -lam / log(1-1/2,2) ) // 2
        # ISD(n_0*p,(n_0-1)*p,n_0*v) / p    attacked code rate 1/n_0
        v_3 = ceil( -lam/ log(1-1/n_0,2)) // n_0

        print(f"lambda: {lam}, n_0: {n_0} t: {t.n()}, v:{v_1.n()} {v_2.n()} {v_3.n()}")
    print("")

lambda: 128, n_0: 2 t: 128.000000000000, v:64.0000000000000 64.0000000000000 64.0000000000000
lambda: 128, n_0: 3 t: 81.0000000000000, v:40.0000000000000 64.0000000000000 73.0000000000000
lambda: 128, n_0: 4 t: 64.0000000000000, v:32.0000000000000 64.0000000000000 77.0000000000000

lambda: 192, n_0: 2 t: 192.000000000000, v:96.0000000000000 96.0000000000000 96.0000000000000
lambda: 192, n_0: 3 t: 122.000000000000, v:61.0000000000000 96.0000000000000 109.000000000000
lambda: 192, n_0: 4 t: 96.0000000000000, v:48.0000000000000 96.0000000000000 115.000000000000

lambda: 256, n_0: 2 t: 256.000000000000, v:128.000000000000 128.000000000000 128.000000000000
lambda: 256, n_0: 3 t: 162.000000000000, v:81.0000000000000 128.000000000000 146.000000000000
lambda: 256, n_0: 4 t: 128.000000000000, v:64.0000000000000 128.000000000000 154.000000000000



In [ ]:
6